# Reporte Ejecutivo — Semanal / Mensual

Genera reportes consolidados a partir de las reuniones procesadas.

1. **Configuracion** — define periodo y conecta con Gemini
2. **Cargar reuniones** — lista todas las reuniones disponibles
3. **Filtrar periodo** — selecciona reuniones del rango deseado
4. **Generar reporte** — consolida y genera el reporte ejecutivo

### Convencion de nombres de archivo
Para que la fecha se detecte automaticamente, nombra tus transcripciones asi:
```
YYMMDD_tema.docx
```
Ejemplo: `260304_daily.docx`, `260228_chinalco_auditoria.docx`

In [ ]:
import sys, json
from pathlib import Path
from datetime import datetime

sys.path.append(str(Path("..") / "src"))

from meeting_assistant import (
    make_client,
    load_all_meetings,
    filter_by_date_range,
    get_week_range,
    get_month_range,
    generate_report,
    report_to_markdown,
)

# -- Rutas --
ENV_PATH    = "../.env"
OUT_DIR     = Path("../outputs")     # JSONs de reuniones individuales
REPORTS_DIR = Path("../reports")     # Reportes ejecutivos (separados)
REPORTS_DIR.mkdir(exist_ok=True)

# -- Tipo de reporte: "semanal" o "mensual" --
REPORT_TYPE = "semanal"

# -- Fecha de referencia (el reporte cubre la semana/mes que contiene esta fecha) --
# Usa None para la fecha actual, o especifica una fecha:
REF_DATE = datetime(2026, 3, 4)  # Cambia segun necesites
# REF_DATE = None  # Usa la fecha actual

# -- Conectar --
client, model, km = make_client(ENV_PATH)
resp = client.models.generate_content(model=model, contents="Di solo OK")
print(f"Modelo  : {model}")
print(f"Conexion: {resp.text.strip()}")
print(f"Reporte : {REPORT_TYPE}")
print(f"Ref date: {REF_DATE or 'hoy'}")
print(f"Salida  : {REPORTS_DIR.resolve()}")

In [2]:
# -- Cargar todas las reuniones procesadas --
all_meetings = load_all_meetings(OUT_DIR)
print(f"Reuniones procesadas: {len(all_meetings)}\n")

for m in all_meetings:
    date_str = m['_date'].strftime('%Y-%m-%d') if m.get('_date') else 'SIN FECHA'
    title = m.get('meeting_title', '?')
    n_tasks = len(m.get('action_items', []))
    n_decisions = len(m.get('decisions', []))
    print(f"  [{date_str}] {title}")
    print(f"    Tareas: {n_tasks} | Decisiones: {n_decisions} | Archivo: {m['_source_file']}")

Reuniones procesadas: 7

  [2026-02-25] Resumen General de Plataforma y Operaciones
    Tareas: 33 | Decisiones: 12 | Archivo: 260225_daily_structured.json
  [2026-03-02] Reunión Plataforma MMM-20260302_171909-Grabación de la reunión
    Tareas: 5 | Decisiones: 5 | Archivo: 260302_plataforma-MMM_structured.json
  [2026-03-03] Reunión Proyecto Chinalco
    Tareas: 6 | Decisiones: 3 | Archivo: 260303_capa-auditoria-Chinalco_structured.json
  [2026-03-03] Desarrollo y Mejoras de Plataforma y FMS
    Tareas: 19 | Decisiones: 12 | Archivo: 260303_feedback-plataforma-MMM_structured.json
  [2026-03-04] Reunión de Mejoras y Automatización de Plataforma HH4M
    Tareas: 14 | Decisiones: 18 | Archivo: 260304_actualizacion-HH4M_structured.json
  [2026-03-04] Reunión CODEa
    Tareas: 23 | Decisiones: 12 | Archivo: 260304_daily-team_structured.json
  [2026-03-06] Reunión de Avances y Estrategia de Proyectos
    Tareas: 36 | Decisiones: 11 | Archivo: 260306_daily-team_structured.json


In [3]:
# -- Calcular rango del periodo --
ref = REF_DATE or datetime.now()

if REPORT_TYPE == "semanal":
    date_from, date_to = get_week_range(ref)
    period_label = f"Semana {date_from.strftime('%d %b')} - {date_to.strftime('%d %b %Y')}"
elif REPORT_TYPE == "mensual":
    date_from, date_to = get_month_range(ref)
    period_label = f"{ref.strftime('%B %Y')}"
else:
    raise ValueError(f"REPORT_TYPE invalido: {REPORT_TYPE}. Usa 'semanal' o 'mensual'.")

print(f"Periodo: {period_label}")
print(f"Rango : {date_from.strftime('%Y-%m-%d')} a {date_to.strftime('%Y-%m-%d')}")

# Filtrar reuniones del periodo
current_meetings = filter_by_date_range(all_meetings, date_from, date_to)
print(f"\nReuniones en este periodo: {len(current_meetings)}")
for m in current_meetings:
    d = m['_date'].strftime('%Y-%m-%d') if m.get('_date') else '?'
    print(f"  [{d}] {m.get('meeting_title', '?')}")

if not current_meetings:
    print("\n** No hay reuniones en este periodo. Ajusta REF_DATE o REPORT_TYPE. **")

Periodo: Semana 02 Mar - 08 Mar 2026
Rango : 2026-03-02 a 2026-03-08

Reuniones en este periodo: 6
  [2026-03-02] Reunión Plataforma MMM-20260302_171909-Grabación de la reunión
  [2026-03-03] Reunión Proyecto Chinalco
  [2026-03-03] Desarrollo y Mejoras de Plataforma y FMS
  [2026-03-04] Reunión de Mejoras y Automatización de Plataforma HH4M
  [2026-03-04] Reunión CODEa
  [2026-03-06] Reunión de Avances y Estrategia de Proyectos


In [4]:
# -- Buscar reuniones del periodo anterior (para comparacion) --
from datetime import timedelta

if REPORT_TYPE == "semanal":
    prev_from = date_from - timedelta(days=7)
    prev_to = date_from - timedelta(seconds=1)
else:
    prev_to = date_from - timedelta(seconds=1)
    prev_from, _ = get_month_range(prev_to)

prev_meetings = filter_by_date_range(all_meetings, prev_from, prev_to)
print(f"Reuniones del periodo anterior ({prev_from.strftime('%Y-%m-%d')} a {prev_to.strftime('%Y-%m-%d')}): {len(prev_meetings)}")
for m in prev_meetings:
    d = m['_date'].strftime('%Y-%m-%d') if m.get('_date') else '?'
    print(f"  [{d}] {m.get('meeting_title', '?')}")

if not prev_meetings:
    print("  (sin datos del periodo anterior -- no se hara comparacion)")

Reuniones del periodo anterior (2026-02-23 a 2026-03-01): 1
  [2026-02-25] Resumen General de Plataforma y Operaciones


In [ ]:
# -- Generar reporte --
if not current_meetings:
    print("No hay reuniones para generar reporte. Ajusta el periodo.")
else:
    print(f"Generando reporte {REPORT_TYPE}: {period_label}")
    print(f"Reuniones: {len(current_meetings)} | Contexto anterior: {'si' if prev_meetings else 'no'}")
    print("Llamando a Gemini...\n")

    report_data = generate_report(
        client, model, km,
        meetings=current_meetings,
        period_label=period_label,
        previous_meetings=prev_meetings or None,
    )

    # Guardar JSON y MD en reports/
    safe_label = period_label.replace(" ", "_").replace("/", "-")
    out_json = REPORTS_DIR / f"reporte_{REPORT_TYPE}_{safe_label}.json"
    out_md   = REPORTS_DIR / f"reporte_{REPORT_TYPE}_{safe_label}.md"

    out_json.write_text(json.dumps(report_data, ensure_ascii=False, indent=2), encoding="utf-8")
    out_md.write_text(report_to_markdown(report_data), encoding="utf-8")

    print(f"Guardado:")
    print(f"  JSON: {out_json}")
    print(f"  MD:   {out_md}")

In [6]:
# -- Vista previa del reporte --
if current_meetings:
    from IPython.display import Markdown, display
    display(Markdown(report_to_markdown(report_data)))

# Reporte Ejecutivo Semanal - 02 al 08 de Marzo de 2026
**Periodo:** Semana 02 Mar - 08 Mar 2026
**Reuniones analizadas:** 6
**Generado:** 2026-03-07 18:24

## Resumen Ejecutivo
La semana ha estado marcada por un intenso desarrollo de plataformas clave y la implementación de automatizaciones estratégicas. Se ha avanzado significativamente en la plataforma Core para gestión de proyectos y en el sistema de reclutamiento HH4M, con énfasis en la integración de IA y la optimización de la experiencia de usuario. El proyecto Chinalco ha iniciado con la definición de requerimientos de datos, mientras que las plataformas de aprendizaje y eventos han mostrado progresos tangibles. Se han tomado decisiones importantes en marketing, incluyendo la contratación de personal y la estrategia de descuentos, aunque persisten desafíos en la coordinación académica y la obtención de credenciales para integraciones críticas.

## Logros Clave del Periodo
- Presentación y demostración de la plataforma 'Core' para gestión de tareas y proyectos, con funcionalidades de seguimiento de tiempos y asignación de dueños.
- Definición de requerimientos de datos GPS y modelo de bloques para el Proyecto Chinalco, incluyendo la decisión de implementar un visualizador 2D para presentación inmediata.
- Implementación de funcionalidades clave en HH4M, como la duplicación de convocatorias, rediseño de paneles administrativos y la opción de automatizar el paso de candidatos entre etapas.
- Avance en la plataforma Code Events con la integración de registro manual de ventas, creación automática de usuarios y conexión con SUNAT (pendiente de credenciales).
- Completado el desarrollo de funcionalidades de la aplicación Comedy Perú, pasando a la fase de pruebas y mejoras de diseño.
- Desarrollo de un dashboard para leads de WhatsApp y corrección de errores en el módulo de ventas de Codea Uni, incluyendo la consulta de DNI.
- Decisión de contratar un diseñador adicional para el equipo de marketing para aliviar la sobrecarga y agilizar la producción de materiales.
- Establecimiento de la política de que todas las postulaciones a Codea se realicen exclusivamente a través de la plataforma Hill Hunting for Mining.

## Progreso por Proyecto

### Plataforma Core / Gestión de Proyectos — _en progreso_

**Avances este periodo:**
- Presentación de la plataforma 'Core' para traqueo de pendientes, proyectos, gestión de timing y calidad de entregables.
- Demostración de vistas principales (feed, clientes, proyectos, tareas, subtareas) y funcionalidades (chat por tarea, adjuntar archivos, editar deadlines, prioridades, estado de tareas).
- Discusión y propuesta de integración con IA para revisión de piezas de diseño, Slack para comunicación y CRM para equipo comercial.
- Definición de jerarquías de roles y proceso de aprobación para el flujo de trabajo de marketing.
- Decisión de implementar edición y eliminación de tareas, subcampañas, subtareas y chats internos por tarea/campaña.
- Avance en la implementación de IA para la generación de productos o entregables finales ('tres capas').

**vs. periodo anterior:** Este periodo marca la introducción y el desarrollo inicial de la plataforma 'Core', enfocada en la gestión de proyectos y la automatización de marketing, lo cual representa una expansión significativa de las funcionalidades generales de plataforma mencionadas previamente (panel administrativo, conexión, carga de cursos/chat).

**Proximos hitos:**
- Oscar preparará un documento con el flujo de trabajo para la creación de un plan de marketing para identificar puntos de automatización.
- Miguel armará la base para los bots y se reunirá con Bill y Cristian para definir datos para el flujo de auditoría.
- Implementar las funcionalidades de editar y eliminar tareas, con restricciones si la tarea está en producción.
- Ajustar el flujo de grillas para que se dirijan a campañas/subcampañas y hereden información.
- Desarrollar un chat interno por pieza/tarea y un chat a nivel de campaña para la comunicación con el cliente.
- Investigar e integrar IA para la autorevisión de piezas de diseño (verificación de colores, posiciones, logos, etc.).

**Bloqueos:**
- La fase de planeamiento de marketing es volátil y laboriosa, lo que podría requerir seguir un flujo manual por ahora.
- La comunicación por WhatsApp es ineficiente y se pierde el hilo de las conversaciones, lo que puede llevar a pérdida de información.
- Existe el riesgo de desvirtuar el core de la plataforma si se adoptan metodologías externas (Scrum, Slack) sin una adaptación cuidadosa.

### Proyecto Chinalco / FMS (Fleet Management System) — _en progreso_

**Avances este periodo:**
- Definición de requerimientos de GPS de cucharas y equipos de acarreo con la frecuencia de logs original.
- Identificación de la necesidad de comprender el modelo de bloques de 10x10m de Chinalco y su uso en el despacho.
- Decisión de implementar un visualizador 2D para la presentación inmediata de avances y un visualizador 3D para la auditoría de viajes en la última capa del FMS.
- Branco debe tener lista la vista en 2D para la presentación de mañana (04 Mar).

**vs. periodo anterior:** Este proyecto no fue mencionado explícitamente en el periodo anterior, lo que indica un inicio o una priorización reciente. El progreso se centra en la fase de levantamiento de requerimientos y diseño inicial.

**Proximos hitos:**
- Enviar un correo a Cristian para solicitar los logs GPS de equipos de acarreo y cucharas con su frecuencia original.
- Coordinar con César para entender el modelo de bloques de Chinalco y el acceso a su data.
- Realizar una reunión para mostrar avances (2D o 3D) y justificar la necesidad de datos.
- Integrar el visualizador 3D en el FMS (capa de auditoría) para la reconstrucción de viajes.
- Cerrar lo que faltaba del FMS y el tema del 'agente'.

**Bloqueos:**
- Dificultad para obtener los logs GPS en su frecuencia original, no la versión 'limpia'.
- Falta de claridad sobre cómo Chinalco define las zonas de mineral económico o desmonte dentro de su modelo de bloques.
- Incompatibilidad potencial entre la estructura del modelo de bloques interno (Mainsay) y el sistema de despacho de Chinalco.
- La reticencia de José al 3D podría limitar la funcionalidad o la visión del proyecto.

### Plataforma HH4M (Hill Hunting for Mining) — _en progreso_

**Avances este periodo:**
- Eliminación del doble logo en la plataforma y desarrollo de un botón 'Lanzar nueva convocatoria' para duplicar puestos.
- Optimización de la visualización de puestos para postulantes y rediseño de paneles de administración (dashboard, pipeline) para reclutadores y psicólogos.
- Cambio de la frecuencia de notificaciones por correo a 'por fase completada' para reducir la cantidad de emails.
- Implementación de la opción de activar/desactivar la automatización para pruebas técnicas y evaluaciones psicológicas.
- Habilitación de la capacidad de seleccionar a múltiples candidatos para un mismo puesto y la funcionalidad de cambio de rol para el administrador.

**vs. periodo anterior:** Este proyecto no fue detallado en el periodo anterior. El progreso actual muestra un enfoque en la mejora de la experiencia de usuario, la automatización de procesos de reclutamiento y la gestión eficiente de puestos y candidatos.

**Proximos hitos:**
- Revisar la lógica para permitir la aprobación de múltiples candidatos a la vez.
- Desarrollar la funcionalidad para configurar formularios como evaluables/no evaluables y ajustar la lógica de evaluación.
- Modificar la vista del 'pipeline' para agrupar por puestos y corregir el cálculo de 'Candidatos pendientes' en el dashboard del psicólogo.
- Ajustar la navegación de la lista de candidatos para mantener el contexto del puesto al volver de un candidato específico.
- Definir si se mantienen los candidatos rechazados en la vista de reclutador, considerando una futura funcionalidad de matching con IA.

**Bloqueos:**
- Reactivar puestos con la misma base de datos de candidatos puede generar confusión.
- El envío de correos por cada prueba completada es 'invasivo' y genera demasiados emails.
- Candidatos calificados pueden ser descartados prematuramente debido a la evaluación incorrecta de formularios de recolección de datos.
- El proceso sigue siendo manual en etapas clave, ralentizando la contratación si no se implementa la automatización.

### Code Events / Codea Mining Fest — _en progreso_

**Avances este periodo:**
- Implementación de la sección de ventas en la plataforma Code Events para Codea Mining Fest.
- Integración de comprobantes y conexión con SUNAT para registrar ventas manuales (pendiente de credenciales).
- Implementación de la creación automática de usuarios y envío de contraseñas al registrar ventas manuales.

**vs. periodo anterior:** Este proyecto no fue mencionado en el periodo anterior. El progreso actual se centra en habilitar las funcionalidades de venta y gestión de usuarios para el evento.

**Proximos hitos:**
- Branco debe preparar un diagrama de flujo del software para la reunión con Chinalco.
- Coordinar una reunión entre Fiorella, Branco y Daniel para capacitación en el uso de la nueva sección de ventas de Codea Mining Fest.
- Obtener las credenciales de SUNAT para la generación de comprobantes.

**Bloqueos:**
- Falta de credenciales de SUNAT para Branco para la generación de comprobantes en Code Events.

### English for Miners — _en progreso_

**Avances este periodo:**
- Harold completó la barra de progreso del estudiante y el orquestador para subir de nivel.
- Se agregaron objetivos y habilidades (listening, writing, reading, vocabulario técnico, gramática) para el progreso.
- Se discutió la implementación de Anki (aprendizaje espaciado) con cartillas agrupadas por categorías.
- Mejoras visuales en el 'orquestador' y gestión de contenido (SSR/Cards).

**vs. periodo anterior:** Este proyecto no fue detallado en el periodo anterior. El progreso actual muestra un avance significativo en la estructura pedagógica y la interfaz de usuario de la plataforma de aprendizaje.

**Proximos hitos:**
- Harold debe terminar la página de English for Miners para el viernes para inspección.
- Implementar la funcionalidad de Anki (aprendizaje espaciado) con carga de tarjetas por el staff y desde escenarios/vocabulario.
- Completar la implementación de la gramática anidada/secuencial y cargar la data correspondiente a los niveles.

### Comedy Perú — _en progreso_

**Avances este periodo:**
- Atheris terminó el desarrollo de funcionalidades de la aplicación.

**vs. periodo anterior:** En el periodo anterior, Harold debía conectar el backend con el frontend y servicios externos. Este periodo, Atheris ha finalizado el desarrollo de funcionalidades, lo que representa un avance sustancial hacia la completitud de la aplicación.

**Proximos hitos:**
- Realizar pruebas necesarias para que la aplicación Comedy Perú esté 100% funcional.
- Mejorar el diseño en el contraste del modo oscuro de Comedy Perú.

### Bot de WhatsApp / Módulo de Ventas (Codea Uni) — _en progreso_

**Avances este periodo:**
- Miguel presentó un dashboard general para leads de WhatsApp y configuración de bootcamps.
- Se corrigieron errores en la generación de ventas y notas de crédito.
- Se añadió la consulta de DNI a la RENIEC, con la limitación de 100 consultas mensuales.
- Se avanzó en el registro de actividades con historial y detalle.

**vs. periodo anterior:** En el periodo anterior, las tareas se centraban en 'continuar seguimiento de bootcamps' y 'emisión de boletas'. Este periodo, se ha avanzado en la automatización y mejora de la gestión de leads y el proceso de ventas a través de un bot y un dashboard, lo que representa una mejora tecnológica en la eficiencia de las operaciones de ventas.

**Proximos hitos:**
- Arreglar el tema de la fecha en el módulo de ventas (toma la hora del servidor).
- Mejorar el formato de las facturas/boletas con el logo de Codea y un diseño más profesional.

**Bloqueos:**
- La API de RENIEC tiene un límite mensual de 100 consultas, lo que requiere una solución alternativa para la consulta de DNI.

### Metamining — _en progreso_

**Avances este periodo:**
- Miguel integró agentes y la grilla de contenido para el plan de 3 meses.
- Se está desarrollando un chatbot para actividades pendientes y temas de la plataforma.

**vs. periodo anterior:** Este proyecto no fue mencionado en el periodo anterior. El progreso actual se enfoca en la integración de IA para la gestión de contenido y la interacción con la plataforma.

### Gestión Académica / Codea UNITECH — _en progreso_

**Avances este periodo:**
- Se definirá la estrategia de cursos de Codea VIP para sacar un curso por mes (Lean Thinking, Metodologías Ágiles).
- Se finalizaron las clases de programación IML con Python para adolescentes y se inició el bootcamp de mantenimiento.
- Se está avanzando con certificados, exámenes y transcripciones de bootcamps.
- Se implementará un examen teórico de máximo 10 preguntas de opción múltiple para los certificados de Codea UNITECH.
- Creación de aulas para todos los cursos antes de que se empiecen a vender.

**vs. periodo anterior:** El periodo anterior identificó problemas con la gestión de certificados (lógica legacy, fechas, asignación), cursos (estudiantes retirados, fechas modificadas) y docentes (entrega de materiales, temarios). Este periodo muestra un avance hacia la estandarización de procesos (exámenes, estrategia VIP) y la gestión de contenido (certificados, transcripciones), aunque persisten desafíos en la coordinación con docentes y la subida de materiales.

**Proximos hitos:**
- Rosmery debe contactar al Ing. Dani para drones y a César Ramírez para metalurgia, y a Bill para docentes de Inti Data.
- Pasar requerimiento a marketing para crear un flyer modelo para anunciar cursos de Codea VIP.
- Agendar reunión con Nilson para revisar la estrategia de cursos de Codea VIP (un curso por mes).
- Subir un examen de máximo 10 preguntas de opción múltiple para los certificados de Codea UNITECH.
- Subir las transcripciones de bootcamps a la plataforma y procesar/subir videos de clases.
- Elaborar y cargar certificados y exámenes en la plataforma (excepto programación).

**Bloqueos:**
- César Ramírez no responde para coordinar sesiones de metalurgia, lo que afecta la programación.
- Problemas con la API de Gemini (error 429) debido a su uso compartido y alto volumen de solicitudes, lo que podría afectar la transcripción de videos.

## Decisiones Clave
- **Priorizar la automatización de los roles de Samantha y Valeria (estrategia y ejecución de marketing) con agentes de IA.** — Optimizará los procesos de marketing, reduciendo la carga manual y acelerando la creación de contenidos. _(Owner: Bill Maquin)_
- **Se implementará un visualizador 2D para el sistema FMS para una presentación inmediata, y el 3D se integrará en la última capa para la auditoría.** — Permitirá mostrar avances rápidamente y asegurar una auditoría detallada de viajes mineros. _(Owner: Branco)_
- **Se implementará un botón 'Lanzar nueva convocatoria' en HH4M para duplicar puestos, heredando todo excepto los registros de candidatos.** — Mejorará la eficiencia en la gestión de convocatorias, reduciendo el tiempo de configuración de nuevos puestos. _(Owner: kanarian080902@outlook.com)_
- **Se aplicará un descuento total del 50% (20% actual + 30% adicional) para los asistentes a la charla informativa.** — Incentivará la participación y la conversión de leads en la charla informativa, impulsando las ventas. _(Owner: Daniel)_
- **Contratar un diseñador adicional para el equipo de marketing.** — Aliviará la sobrecarga del equipo de marketing, acelerando la producción de materiales y campañas. _(Owner: Nilson)_
- **Todas las postulaciones a puestos de Codea deben realizarse exclusivamente a través de la plataforma Hit Hunting for Mining.** — Centralizará el proceso de reclutamiento, mejorando la trazabilidad y la eficiencia en la gestión de candidatos. _(Owner: Wilber)_
- **Se implementará un examen teórico de máximo 10 preguntas de opción múltiple para los certificados de Codea UNITECH.** — Estandarizará el proceso de certificación y asegurará un nivel mínimo de conocimiento para los estudiantes.

## Carga del Equipo
| Miembro | Tareas | Responsabilidades Principales |
|---|---|---|
| Miguel | 18 | Desarrollo de IA y bots para automatización de marketing y gestión de plataforma.; Mejoras en la plataforma Core (gestión de tareas, grillas, chats).; Desarrollo y optimización del módulo de ventas y bot de WhatsApp para Codea Uni.; Integración de agentes y chatbot para Metamining.; Finalización de observaciones en psicología y filtrados, y el pipeline en HH4M. |
| Branco Alberto Villegas Peralta | 14 | Desarrollo de funcionalidades para Code Events (ventas, SUNAT, usuarios).; Implementación de visualizadores 2D/3D para FMS y Proyecto Chinalco.; Preparación de diagramas de flujo y data para reuniones clave.; Desarrollo de nuevos proyectos (Tribu Flying, Ateris, control de acceso discoteca). |
| Rosmery | 17 | Coordinación académica con docentes y definición de estrategias de cursos VIP.; Gestión de certificados, exámenes y transcripciones de bootcamps.; Mantenimiento de plataforma académica (subida de videos, suscripciones, creación de aulas).; Coordinación con marketing para materiales de difusión y con RRHH para especialistas. |
| Harold Mayta | 7 | Desarrollo de la plataforma English for Miners (barra de progreso, orquestador, Anki, gramática).; Mejoras visuales y funcionales en la plataforma de aprendizaje. |
| Daniel Garrido | 4 | Gestión de campañas de remarketing y estrategia de ventas para charlas informativas.; Creación de flyers de descuento y coordinación con Nilson para eventos.; Administración de certificados de cursos. |
| kanarian080902@outlook.com | 14 | Desarrollo backend y frontend para HH4M (aprobación de candidatos, filtros, notificaciones, formularios, automatización, vistas de pipeline y dashboard). |
| Nilson Garrido | 7 | Coordinación de reuniones y estrategias (Hit Hunting, cursos VIP, patrocinios).; Gestión de pagos y recordatorios (Gabriel, Teams).; Coordinación con marketing para flyers y brochures.; Contacto directo con ingenieros para coordinación de clases. |
| Atheris Cipher | 2 | Pruebas y mejoras de diseño en la aplicación Comedy Perú. |
| Oscar | 2 | Preparación de documentación de flujo de trabajo de marketing.; Revisión de entregables de guion con Miguel. |
| Bill Maquin | 5 | Coordinación de reuniones para auditoría y definición de datos.; Priorización de automatizaciones en roles de marketing y planes a largo plazo.; Gestión de proyectos y derivación de tareas (control de acceso discoteca).; Investigación sobre casos de uso de polígonos para auditoría FMS. |
| Wilber | 2 | Gestión de RRHH y marketing (flyers de contratación, eliminación de opciones de postulación). |
| Fiorella Quispe Zanabria | 3 | Marketing a números potenciales y seguimiento de leads.; Coordinación con Nilson para patrocinios. |

## Riesgos y Bloqueos
- La fase de planeamiento de marketing es volátil y laboriosa, lo que podría requerir seguir un flujo manual por ahora.
- La comunicación por WhatsApp es ineficiente y se pierde el hilo de las conversaciones, lo que puede llevar a pérdida de información.
- Dificultad para obtener los logs GPS en su frecuencia original para el Proyecto Chinalco.
- Falta de claridad sobre cómo Chinalco define las zonas de mineral económico o desmonte dentro de su modelo de bloques.
- Incompatibilidad potencial entre la estructura del modelo de bloques interno (Mainsay) y el sistema de despacho de Chinalco.
- La reticencia de José al 3D podría limitar la funcionalidad o la visión del proyecto FMS.
- Reactivar puestos en HH4M con la misma base de datos de candidatos puede generar confusión.
- El envío de correos por cada prueba completada en HH4M es 'invasivo' y genera demasiados emails.
- Candidatos calificados pueden ser descartados prematuramente en HH4M debido a la evaluación incorrecta de formularios de recolección de datos.
- El proceso de reclutamiento en HH4M sigue siendo manual en etapas clave, ralentizando la contratación si no se implementa la automatización.
- Falta de credenciales de SUNAT para Branco para la generación de comprobantes en Code Events.
- La API de RENIEC tiene un límite mensual de 100 consultas, lo que requiere una solución alternativa para la consulta de DNI en el módulo de ventas.
- César Ramírez no responde para coordinar sesiones de metalurgia, lo que afecta la programación académica.
- Problemas con la API de Gemini (error 429) debido a su uso compartido y alto volumen de solicitudes, lo que podría afectar la transcripción de videos académicos.
- El equipo de marketing está sobrecargado, lento y con personal reducido, lo que genera retrasos en los pendientes.
- Riesgo de perder transcripciones de Teams si no se paga la cuenta, como ya ocurrió con transcripciones antiguas.

## Recomendaciones
- Establecer un plan de acción claro para la adquisición de credenciales de SUNAT y la gestión de límites de API (RENIEC, Gemini) para evitar bloqueos en operaciones críticas.
- Formalizar los canales de comunicación interna para la gestión de proyectos y la comunicación de bloqueantes, reduciendo la dependencia de WhatsApp.
- Realizar un seguimiento proactivo con los docentes y especialistas para asegurar la disponibilidad y entrega de materiales académicos en tiempo y forma.
- Priorizar la implementación de las automatizaciones de IA en marketing y reclutamiento para liberar recursos y mejorar la eficiencia operativa.
- Asegurar la coordinación entre los equipos de desarrollo y marketing para la creación y difusión de materiales, especialmente con la nueva contratación de diseñador.

## Perspectiva Proximo Periodo
El próximo periodo se enfocará en la consolidación de las funcionalidades de la plataforma Core, la obtención de datos críticos para el Proyecto Chinalco, y la implementación de las automatizaciones pendientes en HH4M. Se espera un avance significativo en la estrategia de marketing con la integración de IA y la nueva capacidad del equipo. La gestión académica continuará con la definición de cursos VIP y la estandarización de procesos, mientras se resuelven los bloqueos de coordinación con docentes y credenciales de API.

---
_Archivos fuente: 260302_plataforma-MMM_structured.json, 260303_capa-auditoria-Chinalco_structured.json, 260303_feedback-plataforma-MMM_structured.json, 260304_actualizacion-HH4M_structured.json, 260304_daily-team_structured.json, 260306_daily-team_structured.json_